# RQ6: EV User Segmentation and Default-Risk Profiling

This Kaggle notebook applies K-Means clustering, selects K using silhouette score, profiles the segments, and plots default-risk rates.

**Outputs saved to `/kaggle/working/rq6_user_segmentation/`:**
- `RQ6_cluster_selection_silhouette_table.csv`
- `RQ6_user_segmentation_profile_table.csv`
- `RQ6_cluster_assignments.csv`
- `RQ6_default_risk_by_segment_figure.pdf`

**Research question:** What EV-user segments can be identified using charging behavior, financial variables, app usage, and EV characteristics, and how do they differ in default risk?

## Run this cell to generate the actual answer, CSV table, and PDF figure

The notebook automatically searches for the dataset file inside `/kaggle/input`. If it cannot find it, set `DATASET_PATH` manually in the code cell.

In [ ]:

# ============================================================
# Common setup: imports, paths, loading, cleaning, preprocessing
# ============================================================

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    mean_absolute_error, mean_squared_error, r2_score
)

RANDOM_STATE = 42

# Change this manually only if Kaggle cannot auto-detect your file.
# Example:
# DATASET_PATH = "/kaggle/input/ev-charging-behavior-analysis-and-financial-risk/your_file.csv"
DATASET_PATH = None

def file_has_expected_columns(path):
    """
    Checks whether a candidate file looks like the raw EV dataset rather than
    an output table generated by one of the notebooks.
    """
    try:
        if path.lower().endswith(".csv"):
            cols = pd.read_csv(path, nrows=0).columns.astype(str).str.strip().tolist()
        elif path.lower().endswith((".xlsx", ".xls")):
            cols = pd.read_excel(path, nrows=0).columns.astype(str).str.strip().tolist()
        else:
            return False
    except Exception:
        return False

    required = {"High_Default_Risk", "Charging_Efficiency_Index"}
    return required.issubset(set(cols)) and len(cols) >= 10

def find_dataset_file():
    """
    Finds the raw CSV or Excel dataset file automatically.
    Works in Kaggle by searching /kaggle/input first.
    Also avoids accidentally selecting CSV output tables generated by the notebooks.
    """
    if DATASET_PATH is not None and os.path.exists(DATASET_PATH):
        return DATASET_PATH

    roots = ["/kaggle/input", ".", "/mnt/data"]
    extensions = (".csv", ".xlsx", ".xls")
    candidates = []

    for root in roots:
        if not os.path.exists(root):
            continue
        for dirpath, _, filenames in os.walk(root):
            # Skip common output folders when running locally
            if any(part.startswith("rq") and "output" in part for part in dirpath.lower().split(os.sep)):
                continue
            for filename in filenames:
                lower = filename.lower()
                if lower.endswith(extensions):
                    candidates.append(os.path.join(dirpath, filename))

    if not candidates:
        raise FileNotFoundError(
            "No CSV/XLS/XLSX file found. Add the Kaggle dataset to the notebook "
            "or set DATASET_PATH manually."
        )

    # First priority: files that actually contain the required raw dataset columns.
    valid_raw_files = [path for path in candidates if file_has_expected_columns(path)]
    if valid_raw_files:
        keywords = ["ev", "charging", "financial", "risk", "behavior"]
        preferred = [
            path for path in valid_raw_files
            if any(word in os.path.basename(path).lower() for word in keywords)
        ]
        return sorted(preferred or valid_raw_files)[0]

    # Fallback: use filename keywords if header check cannot be completed.
    keywords = ["ev", "charging", "financial", "risk", "behavior"]
    preferred = [
        path for path in candidates
        if any(word in os.path.basename(path).lower() for word in keywords)
    ]

    selected = sorted(preferred or candidates)[0]
    return selected

def load_raw_dataset():
    path = find_dataset_file()
    print(f"Loading dataset from: {path}")

    if path.lower().endswith(".csv"):
        df = pd.read_csv(path)
    elif path.lower().endswith((".xlsx", ".xls")):
        df = pd.read_excel(path)
    else:
        raise ValueError("Unsupported file type. Use CSV, XLSX, or XLS.")

    print(f"Raw dataset shape: {df.shape[0]:,} rows × {df.shape[1]:,} columns")
    return df

def clean_dataset(df):
    """
    Basic cleaning for this EV charging dataset:
    - strips column names and string values
    - removes duplicate rows
    - converts known numeric columns
    - converts impossible negative values into missing values
    - standardizes binary columns where needed
    """
    df = df.copy()
    df.columns = df.columns.astype(str).str.strip()
    df = df.drop_duplicates()

    # Strip text columns
    for col in df.select_dtypes(include=["object"]).columns:
        df[col] = df[col].astype(str).str.strip()
        df[col] = df[col].replace({"nan": np.nan, "None": np.nan, "": np.nan})

    # Standardize binary columns if they came in as text before numeric conversion
    binary_maps = {
        "yes": 1, "y": 1, "true": 1, "high": 1, "risk": 1, "default": 1, "1": 1,
        "no": 0, "n": 0, "false": 0, "low": 0, "non-risk": 0, "non risk": 0, "0": 0
    }
    for col in ["Loan_Taken", "High_Default_Risk"]:
        if col in df.columns and df[col].dtype == "object":
            df[col] = df[col].astype(str).str.lower().map(binary_maps)

    known_numeric = [
        "User_ID", "Age", "Battery_Capacity_kWh", "Charging_Sessions_Per_Month",
        "Avg_Charge_Cost", "Distance_Travelled_Per_Month", "Loan_Taken",
        "Missed_Payments_Last_6M", "Tenure_Months", "App_Usage_Score",
        "Charging_Time_Minutes", "High_Default_Risk", "Charging_Efficiency_Index"
    ]
    for col in known_numeric:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # Treat impossible negative values as missing
    non_negative_cols = [
        "Age", "Battery_Capacity_kWh", "Charging_Sessions_Per_Month",
        "Avg_Charge_Cost", "Distance_Travelled_Per_Month",
        "Missed_Payments_Last_6M", "Tenure_Months", "App_Usage_Score",
        "Charging_Time_Minutes", "Charging_Efficiency_Index"
    ]
    for col in non_negative_cols:
        if col in df.columns:
            df.loc[df[col] < 0, col] = np.nan

    # Keep app score in a practical range if the column exists
    if "App_Usage_Score" in df.columns:
        df.loc[(df["App_Usage_Score"] < 0) | (df["App_Usage_Score"] > 10), "App_Usage_Score"] = np.nan

    return df

def make_output_dir(name):
    output_dir = os.path.join("/kaggle/working" if os.path.exists("/kaggle/working") else ".", name)
    os.makedirs(output_dir, exist_ok=True)
    print(f"Output folder: {output_dir}")
    return output_dir

def split_features_target(df, target_col, drop_cols=None):
    if drop_cols is None:
        drop_cols = []
    if target_col not in df.columns:
        raise KeyError(f"Target column '{target_col}' not found. Available columns: {list(df.columns)}")

    data = df.dropna(subset=[target_col]).copy()

    # Force target to integer binary if classification
    if target_col == "High_Default_Risk":
        data = data[data[target_col].isin([0, 1])].copy()
        data[target_col] = data[target_col].astype(int)

    y = data[target_col]
    X = data.drop(columns=[target_col] + [c for c in drop_cols if c in data.columns])
    return X, y, data

def get_feature_types(X):
    categorical_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
    numeric_cols = [c for c in X.columns if c not in categorical_cols]
    return numeric_cols, categorical_cols

def make_preprocessor(X, scale_numeric=True):
    numeric_cols, categorical_cols = get_feature_types(X)

    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))

    numeric_transformer = Pipeline(steps=numeric_steps)

    try:
        categorical_transformer = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ])
    except TypeError:
        categorical_transformer = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse=False))
        ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_cols),
            ("cat", categorical_transformer, categorical_cols)
        ],
        remainder="drop"
    )
    return preprocessor, numeric_cols, categorical_cols

def get_transformed_feature_names(preprocessor, numeric_cols, categorical_cols):
    feature_names = list(numeric_cols)
    if categorical_cols:
        try:
            ohe = preprocessor.named_transformers_["cat"].named_steps["onehot"]
            cat_names = list(ohe.get_feature_names_out(categorical_cols))
        except Exception:
            cat_names = []
            for col in categorical_cols:
                cat_names.append(col)
        feature_names.extend(cat_names)
    return feature_names

def save_table(df, output_dir, filename):
    path = os.path.join(output_dir, filename)
    df.to_csv(path, index=False)
    print(f"Saved table: {path}")
    return path

def save_figure(output_dir, filename):
    path = os.path.join(output_dir, filename)
    plt.savefig(path, format="pdf", bbox_inches="tight")
    print(f"Saved figure: {path}")
    return path

def style_plot(title, subtitle=None, xlabel=None, ylabel=None):
    plt.title(title, fontsize=15, fontweight="bold", pad=18)
    if subtitle:
        plt.suptitle(subtitle, fontsize=10, y=0.94, color="#3b4f6b")
    if xlabel:
        plt.xlabel(xlabel, fontsize=11)
    if ylabel:
        plt.ylabel(ylabel, fontsize=11)
    plt.grid(axis="y", linestyle="--", alpha=0.35)
    plt.gca().set_axisbelow(True)

def print_dataset_summary(df):
    print("Cleaned dataset shape:", df.shape)
    print("\nColumns:")
    print(list(df.columns))
    print("\nMissing values:")
    display(df.isna().sum().sort_values(ascending=False).to_frame("missing_count").head(20))


# ============================================================
# RQ6: EV user segmentation and default-risk profiling
# ============================================================

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

output_dir = make_output_dir("rq6_user_segmentation")

df_raw = load_raw_dataset()
df = clean_dataset(df_raw)
print_dataset_summary(df)

target_col = "High_Default_Risk"
if target_col not in df.columns:
    raise KeyError(f"Missing target column: {target_col}")

candidate_cluster_features = [
    "Charging_Sessions_Per_Month",
    "Distance_Travelled_Per_Month",
    "Avg_Charge_Cost",
    "Charging_Time_Minutes",
    "Charging_Efficiency_Index",
    "App_Usage_Score",
    "Tenure_Months",
    "Missed_Payments_Last_6M",
    "Income_Level",
    "Loan_Taken",
    "EV_Type",
    "Charger_Working_Status",
    "Battery_Capacity_kWh",
    "Charging_Location_Type"
]

cluster_features = [c for c in candidate_cluster_features if c in df.columns]
if len(cluster_features) < 4:
    raise ValueError("Not enough clustering features found for RQ6.")

analysis_df = df.dropna(subset=[target_col]).copy()
analysis_df = analysis_df[analysis_df[target_col].isin([0, 1])].copy()
analysis_df[target_col] = analysis_df[target_col].astype(int)

X_cluster = analysis_df[cluster_features].copy()

preprocessor, numeric_cols, categorical_cols = make_preprocessor(X_cluster, scale_numeric=True)
X_transformed = preprocessor.fit_transform(X_cluster)

# Select the number of clusters using silhouette score
candidate_k = list(range(2, 7))
silhouette_rows = []

# To keep Kaggle runtime reasonable, calculate silhouette on a sample if needed.
rng = np.random.default_rng(RANDOM_STATE)
if X_transformed.shape[0] > 5000:
    sample_idx = rng.choice(np.arange(X_transformed.shape[0]), size=5000, replace=False)
else:
    sample_idx = np.arange(X_transformed.shape[0])

for k in candidate_k:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=20)
    labels = km.fit_predict(X_transformed)
    score = silhouette_score(X_transformed[sample_idx], labels[sample_idx])
    silhouette_rows.append({"K": k, "Silhouette_Score": score})

silhouette_table = pd.DataFrame(silhouette_rows)
silhouette_table["Silhouette_Score"] = silhouette_table["Silhouette_Score"].round(4)
display(silhouette_table)
save_table(silhouette_table, output_dir, "RQ6_cluster_selection_silhouette_table.csv")

best_k = int(silhouette_table.sort_values("Silhouette_Score", ascending=False).iloc[0]["K"])
print(f"Selected number of clusters using silhouette score: K = {best_k}")

kmeans = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=20)
analysis_df["Cluster"] = kmeans.fit_predict(X_transformed)

def percent_true(series):
    return 100 * np.mean(series)

agg_dict = {
    "User_Share_Percent": ("Cluster", lambda x: 100 * len(x) / len(analysis_df)),
    "Avg_Sessions_Per_Month": ("Charging_Sessions_Per_Month", "mean") if "Charging_Sessions_Per_Month" in analysis_df.columns else ("Cluster", "size"),
    "Avg_Distance_Per_Month": ("Distance_Travelled_Per_Month", "mean") if "Distance_Travelled_Per_Month" in analysis_df.columns else ("Cluster", "size"),
    "Avg_App_Usage_Score": ("App_Usage_Score", "mean") if "App_Usage_Score" in analysis_df.columns else ("Cluster", "size"),
    "Loan_Taken_Percent": ("Loan_Taken", percent_true) if "Loan_Taken" in analysis_df.columns else ("Cluster", "size"),
    "High_Default_Risk_Percent": ("High_Default_Risk", percent_true),
    "Avg_Efficiency_Index": ("Charging_Efficiency_Index", "mean") if "Charging_Efficiency_Index" in analysis_df.columns else ("Cluster", "size"),
    "Avg_Charge_Cost": ("Avg_Charge_Cost", "mean") if "Avg_Charge_Cost" in analysis_df.columns else ("Cluster", "size"),
    "Avg_Missed_Payments": ("Missed_Payments_Last_6M", "mean") if "Missed_Payments_Last_6M" in analysis_df.columns else ("Cluster", "size")
}

profile = analysis_df.groupby("Cluster").agg(**agg_dict).reset_index()

# Add percent not-working charger if available
if "Charger_Working_Status" in analysis_df.columns:
    status = (
        analysis_df.assign(Is_Not_Working=analysis_df["Charger_Working_Status"].astype(str).str.lower().str.contains("not"))
        .groupby("Cluster")["Is_Not_Working"].mean()
        .mul(100)
        .reset_index(name="Not_Working_Charger_Percent")
    )
    profile = profile.merge(status, on="Cluster", how="left")
else:
    profile["Not_Working_Charger_Percent"] = np.nan

# Add low-income percent if available
if "Income_Level" in analysis_df.columns:
    low_income = (
        analysis_df.assign(Is_Low_Income=analysis_df["Income_Level"].astype(str).str.lower().eq("low"))
        .groupby("Cluster")["Is_Low_Income"].mean()
        .mul(100)
        .reset_index(name="Low_Income_Percent")
    )
    profile = profile.merge(low_income, on="Cluster", how="left")
else:
    profile["Low_Income_Percent"] = np.nan

# Create simple business labels from actual profile values
risk_median = profile["High_Default_Risk_Percent"].median()
sessions_median = profile["Avg_Sessions_Per_Month"].median() if "Avg_Sessions_Per_Month" in profile.columns else 0
efficiency_median = profile["Avg_Efficiency_Index"].median() if "Avg_Efficiency_Index" in profile.columns else 0
app_median = profile["Avg_App_Usage_Score"].median() if "Avg_App_Usage_Score" in profile.columns else 0
not_working_median = profile["Not_Working_Charger_Percent"].median()

def label_segment(row):
    risk = row["High_Default_Risk_Percent"]
    sessions = row.get("Avg_Sessions_Per_Month", 0)
    efficiency = row.get("Avg_Efficiency_Index", 0)
    app = row.get("Avg_App_Usage_Score", 0)
    not_working = row.get("Not_Working_Charger_Percent", 0)
    low_income = row.get("Low_Income_Percent", 0)

    if risk >= profile["High_Default_Risk_Percent"].quantile(0.75) and low_income >= profile["Low_Income_Percent"].median():
        return "Cost-sensitive high-risk users"
    if risk >= risk_median and not_working >= not_working_median:
        return "Infrastructure-stressed users"
    if sessions >= sessions_median and risk >= risk_median:
        return "Heavy usage, medium-risk users"
    if sessions < sessions_median and app < app_median:
        return "New low-activity users"
    if efficiency >= efficiency_median and risk < risk_median:
        return "Efficient low-risk users"
    return "Mixed-profile users"

profile["Segment_Label"] = profile.apply(label_segment, axis=1)

# Round and reorder
round_cols = [c for c in profile.columns if c not in ["Cluster", "Segment_Label"]]
for c in round_cols:
    profile[c] = pd.to_numeric(profile[c], errors="coerce").round(2)

profile = profile.sort_values("High_Default_Risk_Percent", ascending=True).reset_index(drop=True)

display(profile)
save_table(profile, output_dir, "RQ6_user_segmentation_profile_table.csv")

# Save cluster assignments too, useful for checking
assignment_cols = ["Cluster"] + [c for c in ["User_ID", "High_Default_Risk"] if c in analysis_df.columns]
assignments = analysis_df[assignment_cols].copy()
save_table(assignments, output_dir, "RQ6_cluster_assignments.csv")

highest_risk = profile.sort_values("High_Default_Risk_Percent", ascending=False).iloc[0]
print(
    f"Actual answer for RQ6: The highest-risk segment is Cluster {highest_risk['Cluster']} "
    f"({highest_risk['Segment_Label']}) with high default risk = "
    f"{highest_risk['High_Default_Risk_Percent']}%."
)

# Figure 6: Default-risk profile by segment
plot_df = profile.sort_values("High_Default_Risk_Percent", ascending=True)
labels = "Cluster " + plot_df["Cluster"].astype(str) + ": " + plot_df["Segment_Label"].astype(str)

plt.figure(figsize=(12, max(6, 0.6 * len(plot_df))))
bars = plt.barh(labels, plot_df["High_Default_Risk_Percent"], color="#2f72c4")

for bar, share in zip(bars, plot_df["User_Share_Percent"]):
    width = bar.get_width()
    plt.text(width + 1, bar.get_y() + bar.get_height()/2, f"{width:.1f}% | share {share:.1f}%", va="center", fontsize=9)

plt.xlabel("High default risk (%)")
plt.title("Figure 6. Default-risk profile across EV user segments", fontsize=15, fontweight="bold", pad=18)
plt.grid(axis="x", linestyle="--", alpha=0.35)
plt.figtext(
    0.5, -0.02,
    "Note: Segments are identified by K-Means clustering and profiled using actual risk rates.",
    ha="center", fontsize=9
)
plt.tight_layout()

save_figure(output_dir, "RQ6_default_risk_by_segment_figure.pdf")
plt.show()
